# FLUX Unified Agent — ARC-AGI-3 Live Test

**Testing FLUX Unified Agent on REAL ARC-AGI-3 games** with the complete cognitive stack from `PHASE_UNIFIED_AGENT_SPEC.md`:

| Component | Purpose |
|-----------|--------|
| **FLUXUnifiedAgent** | Step-aware cognitive loop with VLM-ready architecture |
| **FrameDiffer** | Shows agent what changed after each action |
| **MovementStrategy** | Curiosity-driven navigation for movement games |
| **ClickStrategy** | Smart exploration for click games |
| **SpatialMemory** | Ice (curiosity) + Water (exploration mass) fields |
| **CausalTracker** | Action → effect learning |
| **RuleInducer** | Pattern → rule abstraction |

**Key Innovation:** The agent **SEES what happened** at each step via visual diff between frames, enabling:
1. Understanding the effect of the last action
2. Adapting strategy based on observed changes
3. Automatic handling of different game control types

---
## Cell 1: Environment Setup

In [ ]:
%%time
import os
import sys
from pathlib import Path

# Detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in sys.modules

if IN_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
elif IN_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
else:
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_utils.py').exists():
        ROOT = ROOT.parent
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")
print(f"Root: {ROOT}")
os.chdir(ROOT)

# Add all phase paths (including phase_unified for new agent)
phase_paths = [
    ROOT,
    ROOT / 'phases/phase1',
    ROOT / 'phases/phase2',
    ROOT / 'phases/phase8',
    ROOT / 'phases/phase8_8',
    ROOT / 'phases/phase9_arc',     # Spatial Memory
    ROOT / 'phases/phase_unified',  # Unified Agent (NEW)
]
for p in phase_paths:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"✓ Environment configured with {len(phase_paths)} paths")
print(f"✓ phase_unified path added for FLUXUnifiedAgent")

## Cell 2: Install Dependencies & Load API Key

In [ ]:
# ─────────────────────────────────────────────
# Fix Pillow Version FIRST (Pillow 12.x breaks qwen-vl-utils)
# ─────────────────────────────────────────────
import subprocess
result = subprocess.run(['pip', 'show', 'pillow'], capture_output=True, text=True)
if 'Version: 12.' in result.stdout:
    print("⚠ Pillow 12.x detected - downgrading to 11.0.0...")
    !pip install -q pillow==11.0.0
    print("✓ Pillow downgraded to 11.0.0")
    print("⚠ RESTART KERNEL for changes to take effect!")
else:
    print("✓ Pillow version OK")

# ─────────────────────────────────────────────
# Install Dependencies
# ─────────────────────────────────────────────

# arc-agi toolkit
try:
    import arc_agi
    print("✓ arc-agi already installed")
except ImportError:
    print("Installing arc-agi...")
    !pip install -q arc-agi
    import arc_agi
    print("✓ arc-agi installed")

# VLM dependencies (for Qwen2.5-VL visual reasoning)
try:
    import bitsandbytes
    print("✓ bitsandbytes already installed")
except ImportError:
    print("Installing bitsandbytes (for 4-bit VLM)...")
    !pip install -q bitsandbytes
    print("✓ bitsandbytes installed")

try:
    from qwen_vl_utils import process_vision_info
    print("✓ qwen-vl-utils already installed")
except ImportError:
    print("Installing qwen-vl-utils...")
    !pip install -q qwen-vl-utils
    print("✓ qwen-vl-utils installed")

# ─────────────────────────────────────────────
# Load API Keys from Secrets (Kaggle/Colab first, then .env fallback)
# ─────────────────────────────────────────────

ARC_API_KEY = None
HF_TOKEN = None

# 1. Try Kaggle secrets first
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        ARC_API_KEY = secrets.get_secret('ARC_API_KEY')
        HF_TOKEN = secrets.get_secret('HF_TOKEN')
        if ARC_API_KEY:
            print("✓ ARC_API_KEY loaded from Kaggle secrets")
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from Kaggle secrets")
    except Exception as e:
        print(f"⚠ Kaggle secrets error: {e}")

# 2. Try Colab secrets
elif IN_COLAB:
    try:
        from google.colab import userdata
        ARC_API_KEY = userdata.get('ARC_API_KEY')
        HF_TOKEN = userdata.get('HF_TOKEN')
        if ARC_API_KEY:
            print("✓ ARC_API_KEY loaded from Colab secrets")
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from Colab secrets")
    except Exception as e:
        print(f"⚠ Colab secrets error: {e}")

# 3. Try environment variables
if not ARC_API_KEY:
    ARC_API_KEY = os.environ.get('ARC_API_KEY')
    if ARC_API_KEY:
        print("✓ ARC_API_KEY loaded from environment")

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from environment")

# 4. Fallback to .env file (local development only)
if not ARC_API_KEY or not HF_TOKEN:
    env_file = ROOT / '.env'
    if env_file.exists():
        print("Loading from .env (local fallback)...")
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if not ARC_API_KEY and line.startswith('ARC_API_KEY='):
                    ARC_API_KEY = line.split('=', 1)[1]
                    print("✓ ARC_API_KEY loaded from .env")
                if not HF_TOKEN and line.startswith('hf_token='):
                    HF_TOKEN = line.split('=', 1)[1]
                    print("✓ HF_TOKEN loaded from .env")

# Set environment variables for libraries that need them
if ARC_API_KEY:
    os.environ['ARC_API_KEY'] = ARC_API_KEY
    print(f"  ARC_API_KEY: {ARC_API_KEY[:8]}...{ARC_API_KEY[-4:]}")
else:
    print("✗ ARC_API_KEY not found!")
    print("  Add to Kaggle/Colab secrets or set as environment variable")

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print(f"  HF_TOKEN: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")
else:
    print("⚠ HF_TOKEN not found (VLM download may fail for gated models)")

## Cell 3: Device Setup & Load Model

Loads `Flux-multi-model.flx` (most capable multi-modal model) with fallback to `Flux-ARC.flx` → `Flux-Base.flx`

In [ ]:
%%time
import torch
import numpy as np
from datetime import datetime

# Device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load Flux-multi-model.flx (most capable model) or download from HuggingFace
CHECKPOINTS_DIR = ROOT / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

# Priority order: multi-model > ARC > Base
MODEL_PRIORITY = [
    'Flux-multi-model.flx',  # Most capable - multi-modal trained
    'Flux-ARC.flx',          # ARC-specific cognitive layer
    'Flux-Base.flx',         # Fallback
]

model_path = None
for model_name in MODEL_PRIORITY:
    candidate = CHECKPOINTS_DIR / model_name
    if candidate.exists():
        model_path = candidate
        print(f"✓ Found local: {model_name}")
        break

if model_path is None:
    print("Downloading Flux-multi-model.flx from HuggingFace...")
    from huggingface_hub import hf_hub_download
    
    for model_name in MODEL_PRIORITY:
        try:
            hf_hub_download(
                repo_id='UnseenGAP/FLUX',
                filename=f'checkpoints/{model_name}',
                local_dir=str(ROOT),
                token=HF_TOKEN,
            )
            model_path = CHECKPOINTS_DIR / model_name
            print(f"✓ Downloaded {model_name}")
            break
        except Exception as e:
            print(f"⚠ {model_name} download failed: {e}")
            continue
    
    if model_path is None:
        raise FileNotFoundError("No FLUX checkpoint available!")

# Load model
print(f"\nLoading {model_path.name}...")
flux_model = torch.load(str(model_path), map_location='cpu', weights_only=False)

print(f"✓ Loaded FLUX model")
print(f"  Format: {flux_model.get('format', 'unknown')}")
print(f"  Version: {flux_model.get('version', 'unknown')}")
print(f"  Components: {list(flux_model.get('components', {}).keys())[:8]}...")

## Cell 4: Initialize Cognitive Layer

Loads cognitive layer from checkpoint if available, otherwise creates fresh components.

In [ ]:
%%time
print("Initializing Cognitive Layer...")
print("=" * 60)

# Import cognitive components
from causal_tracker import CausalTracker
from rule_inducer import RuleInducer
from goal_planner import GoalPlanner, GoalType
from perception_field import PerceptionField
from arc_interface import GameAction, GameState, ActionCommand

# Check if model has cognitive layer built-in
has_cognitive = 'cognitive' in flux_model

if has_cognitive:
    print("Loading cognitive layer from checkpoint...")
    cog = flux_model['cognitive']
    
    ct_config = cog['causal_tracker']['config']
    causal_tracker = CausalTracker(
        max_history=ct_config.get('max_history', 1000),
        device='cpu',
    )
    causal_tracker.load_state_dict(cog['causal_tracker']['state_dict'])
    
    ri_config = cog['rule_inducer']['config']
    rule_inducer = RuleInducer(
        causal_tracker=causal_tracker,
        min_observations=ri_config.get('min_observations', 2),
        min_confidence=ri_config.get('min_confidence', 0.5),
    )
    rule_inducer.load_state_dict(cog['rule_inducer']['state_dict'])
    
    goal_planner = GoalPlanner()
    goal_planner.load_state_dict(cog['goal_planner']['state_dict'])
    
    pf_config = cog['perception_field']['config']
    perception_field = PerceptionField(
        max_size=pf_config.get('max_size', 64),
        fovea_radius=pf_config.get('fovea_radius', 5),
    )
    perception_field.load_state_dict(cog['perception_field']['state_dict'])
    
    print(f"✓ Loaded cognitive layer from {model_path.name}")
else:
    print(f"Creating fresh cognitive layer (not in {model_path.name})...")
    causal_tracker = CausalTracker(max_history=1000, device='cpu')
    rule_inducer = RuleInducer(causal_tracker=causal_tracker)
    goal_planner = GoalPlanner()
    perception_field = PerceptionField(max_size=64, fovea_radius=5)
    print("✓ Created fresh cognitive layer")

print(f"\nCognitive Components Ready:")
print(f"  ✓ CausalTracker (links: {len(causal_tracker.causal_links)})")
print(f"  ✓ RuleInducer (rules: {len(rule_inducer.rules)})")
print(f"  ✓ GoalPlanner")
print(f"  ✓ PerceptionField (fovea_r={perception_field.fovea_radius})")

## Cell 5: Initialize Spatial Memory (Ice & Water)

In [ ]:
%%time
print("Initializing Spatial Memory (Ice & Water)...")
print("=" * 60)

from spatial_memory import SpatialMemory, NavigationTarget
import torch
import torch.nn.functional as F

# Create spatial memory instance
spatial_memory = SpatialMemory(
    max_size=64,          # ARC grids up to 64x64
    feature_dim=64,
    curiosity_threshold=0.1,
    device='cpu',
)

# ─────────────────────────────────────────────
# Monkey-patch encode_cell to handle colors >= 10
# ARC-AGI-3 may use extended color palette
# ─────────────────────────────────────────────
_original_encode_cell = spatial_memory.encode_cell

def _safe_encode_cell(self, color: int):
    """Safe encode_cell that clamps colors to 0-9."""
    safe_color = max(0, min(9, color))  # Clamp to valid range
    one_hot = F.one_hot(torch.tensor(safe_color), num_classes=10).float()
    one_hot = one_hot.to(self.device)
    return self.cell_encoder(one_hot)

# Bind the method
import types
spatial_memory.encode_cell = types.MethodType(_safe_encode_cell, spatial_memory)

print("✓ SpatialMemory initialized")
print("✓ Patched encode_cell to handle colors >= 10 (clamped to 0-9)")
print(f"  Max grid size: 64x64")
print(f"  Feature dim: 64")
print(f"  Curiosity threshold: 0.1")
print()
print("Dual-Field System:")
print("  ❄ CURIOSITY ICE — Highlights interesting locations")
print("  ≈ EXPLORATION MASS — Tracks visited locations")

## Cell 6: Initialize Grid Encoder (GridToWave)

In [ ]:
%%time
print("Initializing GridToWave Encoder...")
print("=" * 60)

from grid_adapters import GridToWave, WaveToGrid

WAVE_DIM = 432

# Initialize encoder with LARGER max_size for ARC-AGI-3 64x64 grids
# and num_colors=16 in case ARC-AGI-3 uses extended color palette
grid_to_wave = GridToWave(
    wave_dim=WAVE_DIM,
    max_size=64,      # ARC-AGI-3 uses 64x64 grids
    num_colors=16,    # Extended color range (0-15) for safety
    device='cpu'
)

# Try to load trained weights from model
if 'grid_adapters' in flux_model and 'encoder' in flux_model['grid_adapters']:
    try:
        grid_to_wave.load_state_dict(flux_model['grid_adapters']['encoder'], strict=False)
        print("✓ Loaded trained GridToWave from checkpoint (partial match)")
    except Exception as e:
        print(f"⚠ Could not load trained weights: {e}")
        print("  Using fresh GridToWave")
else:
    print("⚠ Using fresh GridToWave (no trained weights)")

grid_to_wave.eval()

# Test encoding with 64x64 grid
test_grid = [[0]*64 for _ in range(64)]
test_grid[32][32] = 5
with torch.no_grad():
    test_wave = grid_to_wave.encode(test_grid, mode='holistic')
    print(f"\nTest encoding: 64x64 grid -> [{WAVE_DIM}] wave")
    print(f"  Wave norm: {test_wave.norm().item():.4f}")

## Cell 7: Initialize FLUX Unified Agent with Vision

Loads **Qwen2.5-VL-3B-Instruct** for visual reasoning (requires CUDA GPU):
- 4-bit quantized to ~1.1GB VRAM
- Renders game grid as image with ice overlay
- VLM reasons about patterns and suggests actions

Falls back to **strategy-only mode** on CPU/MPS:
- Movement: follows curiosity gradients (ice)
- Click: explores high-curiosity cells systematically

Both modes use the step-aware cognitive loop from `PHASE_UNIFIED_AGENT_SPEC.md`.

In [ ]:
%%time
print("Initializing FLUX Unified Agent with Vision (Qwen2.5-VL)...")
print("=" * 60)

# Import unified agent components
from unified_agent import FLUXUnifiedAgent, create_unified_agent
from frame_differ import FrameDiffer
from strategies import get_control_scheme, MovementStrategy, ClickStrategy
from game_loop import play_game_unified, GameResult

# ─────────────────────────────────────────────
# Load Qwen2.5-VL for Visual Reasoning
# ─────────────────────────────────────────────

VLM_READY = False
vlm_model = None
vlm_processor = None

# Only load VLM on CUDA (requires GPU)
if device == 'cuda':
    print("\nLoading Qwen2.5-VL-3B-Instruct for visual reasoning...")
    
    try:
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
        from transformers import BitsAndBytesConfig
        
        VLM_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
        
        # 4-bit quantization for memory efficiency (~1.1GB VRAM)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
        
        vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            VLM_MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )
        
        vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
        
        VLM_READY = True
        print(f"✓ Qwen2.5-VL loaded! (4-bit quantized)")
        print(f"  Model: {VLM_MODEL_ID}")
        
        # Quick test
        print("\n  Testing VLM...")
        from PIL import Image
        test_img = Image.new('RGB', (64, 64), color='blue')
        test_messages = [{"role": "user", "content": [
            {"type": "image", "image": test_img},
            {"type": "text", "text": "What color is this image? Reply in one word."}
        ]}]
        test_text = vlm_processor.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
        from qwen_vl_utils import process_vision_info
        test_img_inputs, _ = process_vision_info(test_messages)
        test_inputs = vlm_processor(text=[test_text], images=test_img_inputs, return_tensors="pt").to(vlm_model.device)
        with torch.no_grad():
            test_out = vlm_model.generate(**test_inputs, max_new_tokens=10)
        test_response = vlm_processor.batch_decode(test_out[:, test_inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
        print(f"  Test response: '{test_response.strip()}'")
        print(f"  ✓ VLM working!")
        
    except Exception as e:
        print(f"⚠ VLM loading failed: {e}")
        print("  Falling back to strategy-only mode")
        VLM_READY = False
else:
    print(f"\n⚠ VLM requires CUDA (current: {device})")
    print("  Agent will use strategy-only mode (no visual reasoning)")
    print("  For VLM: run on Kaggle with GPU T4 x2 or P100")

# ─────────────────────────────────────────────
# Create Unified Agent
# ─────────────────────────────────────────────

print("\n" + "-" * 60)
print("Creating FLUXUnifiedAgent...")

agent = create_unified_agent(
    spatial_memory=spatial_memory,
    causal_tracker=causal_tracker,
    model=vlm_model if VLM_READY else None,
    processor=vlm_processor if VLM_READY else None,
    device=device,
    verbose=True,
)

print("\n✓ FLUXUnifiedAgent created with:")
print(f"  - SpatialMemory (ice/water): {spatial_memory is not None}")
print(f"  - CausalTracker: {causal_tracker is not None}")
print(f"  - VLM (Qwen2.5-VL): {agent.vlm_available}")
print()

if agent.vlm_available:
    print("🔮 VISION MODE ENABLED")
    print("   Agent can SEE the game grid as an image!")
    print("   - Renders grid with ice overlay (cyan = curiosity)")
    print("   - VLM reasons about patterns and suggests actions")
    print("   - Falls back to strategy if VLM fails")
else:
    print("📊 STRATEGY MODE (no VLM)")
    print("   Agent uses heuristic strategies:")
    print("   - MOVEMENT: follows curiosity gradients")
    print("   - CLICK: explores high-ice cells systematically")

print()
print("Capabilities:")
print("  1. Step-aware cognitive loop (sees what changed)")
print("  2. Control-specific strategies (movement/click)")
print("  3. Frame differ (tracks action → effect)")
print("  4. Spatial memory (ice = curiosity, water = explored)")

## Cell 8: Test Connection to ARC API

In [ ]:
import requests

print("Testing ARC-AGI-3 API Connection...")
print("=" * 60)

ROOT_URL = "https://three.arcprize.org"

# Create session
session = requests.Session()
session.headers.update({
    "X-API-Key": ARC_API_KEY,
    "Accept": "application/json"
})

# Get games list
response = session.get(f"{ROOT_URL}/api/games")

if response.status_code == 200:
    games = response.json()
    print(f"\u2713 Connected! Found {len(games)} games")
    print(f"\nAvailable games:")
    for game in games[:10]:
        print(f"  - {game['game_id']}: {game.get('title', 'N/A')}")
    if len(games) > 10:
        print(f"  ... and {len(games) - 10} more")
else:
    print(f"\u2717 API Error: {response.status_code}")
    print(response.text)

## Cell 8.5: Initialize Game Memory System (Persistent Learning)

The **Game Memory System** enables real-time learning that persists across runs:

| Component | Purpose |
|-----------|---------|
| `game_memory` | Stores per-game learning (layouts, strategies, outcomes) |
| `promising_paths` | Directions/actions that led to progress |
| `failed_paths` | Actions to avoid (led to no progress) |
| `discovered_rules` | Rules the LLM creates on-the-fly |
| `action_history` | Complete trace for replay analysis |

**Key Behaviors:**
- On first play: Explore freely, record everything
- On replay: Load previous experience, try different strategies
- After each action: Record causal link, update what works
- End of game: Induce rules, save discoveries

In [ ]:
# =============================================================================
# GAME MEMORY SYSTEM — Persistent Learning Across Runs
# =============================================================================

from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime
import json

@dataclass
class GameExperience:
    """Stores everything learned from a single game."""
    game_id: str
    game_type: str  # e.g., "arc" or guessed name
    first_seen: str
    last_played: str
    play_count: int = 0
    
    # What we know about this game
    grid_size: Tuple[int, int] = (3, 3)
    unique_colors: List[int] = field(default_factory=list)
    grid_signature: str = ""  # Hash to recognize same game
    
    # Learning history
    actions_tried: List[Dict[str, Any]] = field(default_factory=list)
    successful_actions: List[Dict[str, Any]] = field(default_factory=list)
    failed_actions: List[Dict[str, Any]] = field(default_factory=list)
    
    # Strategies
    promising_directions: List[str] = field(default_factory=list)
    avoid_directions: List[str] = field(default_factory=list)
    
    # Discovered patterns
    local_rules: List[str] = field(default_factory=list)  # Rules specific to this game
    llm_insights: List[str] = field(default_factory=list)  # LLM-generated observations
    
    # Outcomes
    best_score: float = 0.0
    solved: bool = False
    solution_path: Optional[List[Dict]] = None

class GameMemory:
    """
    Persistent memory across games and sessions.
    Enables: "If it ever plays the game again, it should know what it did last time."
    """
    
    def __init__(self, flux_model=None):
        self.games: Dict[str, GameExperience] = {}
        self.global_rules: List[str] = []  # Rules that apply across games
        self.discovery_log: List[Dict] = []  # Real-time discoveries
        self.flux_model = flux_model
        
        # Load existing game memory from FLUX model if available
        self._load_from_model()
    
    def _compute_grid_signature(self, grid) -> str:
        """Create a hash signature to recognize identical grids."""
        import hashlib
        if hasattr(grid, 'tolist'):
            grid_list = grid.tolist()
        elif isinstance(grid, list):
            grid_list = grid
        else:
            return ""
        grid_str = str(grid_list)
        return hashlib.md5(grid_str.encode()).hexdigest()[:12]
    
    def _load_from_model(self):
        """Load previous game experiences from FLUX model."""
        if self.flux_model is None:
            print("  ℹ No FLUX model provided — starting fresh")
            return
        
        memory_data = self.flux_model.get('game_memory', {})
        if isinstance(memory_data, dict):
            games_data = memory_data.get('games', {})
            for game_id, exp_dict in games_data.items():
                try:
                    self.games[game_id] = GameExperience(**exp_dict)
                except Exception as e:
                    print(f"  ⚠ Could not restore game {game_id}: {e}")
            
            self.global_rules = memory_data.get('global_rules', [])
            print(f"  ✓ Loaded {len(self.games)} previous game experiences")
            print(f"  ✓ Loaded {len(self.global_rules)} global rules")
        else:
            print("  ℹ No previous game memory found")
    
    def save_to_model(self):
        """Save all game experiences back to FLUX model."""
        if self.flux_model is None:
            print("  ⚠ No FLUX model — cannot save")
            return
        
        games_dict = {}
        for game_id, exp in self.games.items():
            exp_dict = asdict(exp)
            # Convert tuples to lists for JSON serialization
            if isinstance(exp_dict.get('grid_size'), tuple):
                exp_dict['grid_size'] = list(exp_dict['grid_size'])
            games_dict[game_id] = exp_dict
        
        self.flux_model.set('game_memory', {
            'games': games_dict,
            'global_rules': self.global_rules,
            'discovery_log': self.discovery_log[-100:],  # Keep last 100 discoveries
            'last_saved': datetime.now().isoformat()
        })
        print(f"  ✓ Saved {len(self.games)} game experiences to model")
    
    def start_game(self, game_id: str, input_grid) -> GameExperience:
        """Called when starting a new game. Returns existing experience or creates new."""
        
        signature = self._compute_grid_signature(input_grid)
        
        # Check if we've seen this exact grid before (by signature)
        for gid, exp in self.games.items():
            if exp.grid_signature == signature:
                print(f"\n  🔁 RECOGNIZED GAME! We've played this {exp.play_count} times before!")
                if exp.successful_actions:
                    print(f"     ✓ Previous successes: {len(exp.successful_actions)}")
                    print(f"     → Will try: {exp.promising_directions[:3]}")
                if exp.failed_actions:
                    print(f"     ✗ Previous failures: {len(exp.failed_actions)}")
                    print(f"     → Will avoid: {exp.avoid_directions[:3]}")
                if exp.local_rules:
                    print(f"     📋 Known rules for this game:")
                    for rule in exp.local_rules[:3]:
                        print(f"        • {rule}")
                
                exp.play_count += 1
                exp.last_played = datetime.now().isoformat()
                return exp
        
        # New game we haven't seen
        if hasattr(input_grid, 'shape'):
            h, w = input_grid.shape[:2]
        elif isinstance(input_grid, list) and len(input_grid) > 0:
            h, w = len(input_grid), len(input_grid[0]) if input_grid else 0
        else:
            h, w = 3, 3
        
        unique_colors = []
        if hasattr(input_grid, 'flatten'):
            unique_colors = list(set(input_grid.flatten().tolist()))
        
        exp = GameExperience(
            game_id=game_id,
            game_type="arc",
            first_seen=datetime.now().isoformat(),
            last_played=datetime.now().isoformat(),
            play_count=1,
            grid_size=(h, w),
            unique_colors=unique_colors,
            grid_signature=signature
        )
        
        self.games[game_id] = exp
        print(f"\n  🆕 NEW GAME: {game_id}")
        print(f"     Grid: {h}x{w}, Colors: {unique_colors}")
        
        return exp
    
    def record_action(self, game_exp: GameExperience, action: Dict, 
                      before_grid, after_grid, reward: float = 0.0):
        """Record what happened after an action."""
        
        action_record = {
            'action': action,
            'timestamp': datetime.now().isoformat(),
            'reward': reward,
            'grid_changed': not self._grids_equal(before_grid, after_grid)
        }
        
        # Track in general history
        game_exp.actions_tried.append(action_record)
        
        # Categorize as successful or failed
        if reward > 0 or action_record['grid_changed']:
            game_exp.successful_actions.append(action_record)
            direction = action.get('direction', action.get('type', 'unknown'))
            if direction not in game_exp.promising_directions:
                game_exp.promising_directions.append(direction)
            print(f"     ✓ Action worked: {direction} → reward={reward:.2f}")
        else:
            game_exp.failed_actions.append(action_record)
            direction = action.get('direction', action.get('type', 'unknown'))
            if direction not in game_exp.avoid_directions:
                game_exp.avoid_directions.append(direction)
        
        return action_record
    
    def _grids_equal(self, grid1, grid2) -> bool:
        """Check if two grids are identical."""
        try:
            import numpy as np
            if hasattr(grid1, '__array__') and hasattr(grid2, '__array__'):
                return np.array_equal(np.array(grid1), np.array(grid2))
            return str(grid1) == str(grid2)
        except:
            return str(grid1) == str(grid2)
    
    def add_rule(self, game_exp: GameExperience, rule: str, is_global: bool = False):
        """Add a rule discovered during gameplay."""
        if is_global:
            if rule not in self.global_rules:
                self.global_rules.append(rule)
                print(f"  📋 NEW GLOBAL RULE: {rule}")
        else:
            if rule not in game_exp.local_rules:
                game_exp.local_rules.append(rule)
                print(f"  📋 NEW LOCAL RULE: {rule}")
    
    def add_llm_insight(self, game_exp: GameExperience, insight: str):
        """Add an LLM-generated insight about this game."""
        if insight not in game_exp.llm_insights:
            game_exp.llm_insights.append(insight)
            self.discovery_log.append({
                'type': 'llm_insight',
                'game_id': game_exp.game_id,
                'insight': insight,
                'timestamp': datetime.now().isoformat()
            })
    
    def get_strategy_prompt(self, game_exp: GameExperience) -> str:
        """Generate a prompt for the LLM based on what we know about this game."""
        
        prompt_parts = [f"Playing game: {game_exp.game_id} (played {game_exp.play_count}x)"]
        prompt_parts.append(f"Grid: {game_exp.grid_size[0]}x{game_exp.grid_size[1]}")
        
        if game_exp.play_count > 1:
            prompt_parts.append("\n--- PREVIOUS EXPERIENCE ---")
            
            if game_exp.promising_directions:
                prompt_parts.append(f"✓ PROMISING: {', '.join(game_exp.promising_directions[:5])}")
            
            if game_exp.avoid_directions:
                prompt_parts.append(f"✗ AVOID: {', '.join(game_exp.avoid_directions[:5])}")
            
            if game_exp.local_rules:
                prompt_parts.append("Known rules:")
                for rule in game_exp.local_rules[:5]:
                    prompt_parts.append(f"  • {rule}")
            
            if game_exp.llm_insights:
                prompt_parts.append("Previous insights:")
                for insight in game_exp.llm_insights[-3:]:
                    prompt_parts.append(f"  • {insight}")
        
        if self.global_rules:
            prompt_parts.append("\n--- GLOBAL RULES (from all games) ---")
            for rule in self.global_rules[:5]:
                prompt_parts.append(f"  • {rule}")
        
        return "\n".join(prompt_parts)

# Initialize game memory with flux model
game_memory = GameMemory(flux_model=flux_model)

print("\n" + "="*60)
print("GAME MEMORY SYSTEM INITIALIZED")
print("="*60)
print(f"  Games remembered: {len(game_memory.games)}")
print(f"  Global rules: {len(game_memory.global_rules)}")
print(f"  Ready for real-time learning!")

## Cell 9: Play Single Game with FLUX — Real-Time Learning

The enhanced `play_game_with_flux()` now implements **real-time learning**:

| Feature | Implementation |
|---------|---------------|
| **Causal Recording** | Every action→effect stored as causal link |
| **LLM Rule Creation** | VLM creates rules on-the-fly when discoveries happen |
| **Game Memory** | Remembers games played for smarter replays |
| **Strategy Selection** | Uses previous experience to avoid failures |
| **Periodic Induction** | Every 10 actions, analyzes patterns for rules |

**On First Play**: Explore freely, record everything learned  
**On Replay**: Load previous experience, try different strategies, apply learned rules

In [ ]:
# =============================================================================
# PLAY GAME WITH REAL-TIME LEARNING
# =============================================================================
# This version:
# - Records every action → effect as a causal link
# - Induces rules on-the-fly when patterns emerge
# - Uses LLM to reason about discoveries and create new rules
# - Remembers games played (via GameMemory) for future replays
# - Learns from what worked and avoids what didn't
# =============================================================================

def llm_create_rule(observation: str, causal_links: list) -> Optional[str]:
    """
    Use the VLM to reason about observations and create rules on-the-fly.
    Returns a rule string or None if no rule can be induced.
    """
    if not vlm_available or len(causal_links) < 3:
        return None
    
    # Build prompt from recent causal observations
    recent_links = causal_links[-10:]  # Last 10 actions
    links_desc = "\n".join([
        f"  Action '{l.get('action', 'unknown')}' → Changed {l.get('changed_cells', 0)} cells"
        for l in recent_links if isinstance(l, dict)
    ])
    
    prompt = f"""You are a rule engine for an ARC puzzle agent. Based on these observations:

{observation}

Recent action history:
{links_desc}

If you notice a pattern, state it as a SHORT rule (max 15 words).
Format: IF [condition] THEN [action/result]
If no clear pattern, respond with just "NO_RULE".
"""
    
    try:
        # Use VLM for text reasoning
        response = vlm_pipe(
            prompt,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=vlm_pipe.tokenizer.eos_token_id,
        )
        result = response[0]['generated_text'].split(prompt)[-1].strip()
        
        if "NO_RULE" in result or len(result) < 10:
            return None
        
        # Clean up result
        rule = result.split('\n')[0].strip()
        if len(rule) > 100:
            rule = rule[:100]
        return rule
    except Exception as e:
        print(f"  ⚠ LLM rule creation failed: {e}")
        return None

def play_game_with_flux(game_id: str, max_actions: int = 100, verbose: bool = True):
    """
    Play a single ARC-AGI-3 game with REAL-TIME LEARNING.
    
    Implements:
    - Records causal links after EVERY action
    - Induces rules every 10 actions
    - Uses LLM reasoning when discoveries happen
    - Remembers this game for future plays
    - Avoids previously failed actions when replaying
    """
    global causal_tracker, rule_inducer, game_memory
    
    print(f"\n{'='*60}")
    print(f"🎮 PLAYING: {game_id} with REAL-TIME LEARNING")
    print(f"{'='*60}")
    
    # Reset agent
    agent.reset()
    agent.verbose = verbose
    
    # Open scorecard
    response = session.post(
        f"{ROOT_URL}/api/scorecard/open",
        json={"tags": ["flux-unified", "real-time-learning", game_id]}
    )
    if response.status_code != 200:
        print(f"✗ Failed to open scorecard: {response.text}")
        return None
    
    card_id = response.json()["card_id"]
    print(f"  Scorecard: {card_id}")
    
    # Reset game
    response = session.post(
        f"{ROOT_URL}/api/cmd/RESET",
        json={"game_id": game_id, "card_id": card_id}
    )
    if response.status_code != 200:
        print(f"✗ RESET failed: {response.text}")
        return None
    
    game_data = response.json()
    guid = game_data["guid"]
    state = game_data["state"]
    score = game_data.get("score", 0)
    frame = game_data.get("frame", [[]])
    available_actions = game_data.get("available_actions", [1, 2, 3, 4, 5])
    
    # Convert to list of ints if needed
    if available_actions and hasattr(available_actions[0], 'value'):
        available_actions = [a.value for a in available_actions]
    
    # =========================================================================
    # REAL-TIME LEARNING: Start game memory tracking
    # =========================================================================
    input_grid = np.array(frame) if isinstance(frame, list) else frame
    game_exp = game_memory.start_game(game_id, input_grid)
    
    # Build strategy prompt from previous experience
    strategy_prompt = game_memory.get_strategy_prompt(game_exp)
    print(f"\n--- Strategy Context ---\n{strategy_prompt}\n")
    
    # Initialize unified agent for this game
    agent.start_game(game_id, frame, available_actions)
    
    print(f"  Initial state: {state}, Score: {score}")
    print(f"  Control scheme: {agent.control_scheme}")
    print(f"  Available actions: {available_actions}")
    
    # =========================================================================
    # MAIN PLAY LOOP WITH REAL-TIME LEARNING
    # =========================================================================
    actions_taken = 0
    level_wins = 0
    last_action = None
    discoveries_this_game = 0
    causal_links_this_game = []
    
    while state == "NOT_FINISHED" and actions_taken < max_actions:
        # Store grid state BEFORE action
        before_grid = np.array(frame) if isinstance(frame, list) else frame.copy()
        
        # Update available actions
        if available_actions and hasattr(available_actions[0], 'value'):
            available_actions = [a.value for a in available_actions]
        
        # =====================================================================
        # STRATEGY: Use previous experience on replay
        # =====================================================================
        avoid_actions = []
        prefer_actions = []
        
        if game_exp.play_count > 1:
            # Get actions to avoid from failed attempts
            for failed in game_exp.failed_actions[-20:]:
                action_type = failed['action'].get('type', failed['action'].get('direction'))
                if action_type:
                    avoid_actions.append(action_type)
            
            # Get actions to prefer from successful attempts
            for success in game_exp.successful_actions[-10:]:
                action_type = success['action'].get('type', success['action'].get('direction'))
                if action_type:
                    prefer_actions.append(action_type)
        
        try:
            # Agent step with strategy hints
            action, action_data, reasoning = agent.step(
                frame=frame,
                available_actions=available_actions,
                last_action=last_action,
            )
            
            action_name = ['RESET', 'UP', 'DOWN', 'LEFT', 'RIGHT', 'INTERACT', 'CLICK', 'UNDO'][action] if action < 8 else f'ACTION{action}'
            
            # If replaying, potentially override with previous knowledge
            if game_exp.play_count > 1 and action_name in avoid_actions and prefer_actions:
                print(f"  ⚠ {action_name} previously failed - trying preferred action")
                # Could add more sophisticated action selection here
            
            if verbose and actions_taken % 10 == 0:
                print(f"\n  Step {actions_taken}: action={action_name}, pos={agent.current_position}")
            
        except Exception as e:
            print(f"✗ Agent error at step {actions_taken}: {e}")
            import traceback
            traceback.print_exc()
            break
        
        # Build request
        request_data = {
            "game_id": game_id,
            "card_id": card_id,
            "guid": guid,
        }
        request_data.update(action_data)
        
        # Take action
        action_endpoint = f"ACTION{action}" if action > 0 else "RESET"
        response = session.post(
            f"{ROOT_URL}/api/cmd/{action_endpoint}",
            json=request_data
        )
        
        if response.status_code != 200:
            print(f"✗ Action failed: {response.text}")
            break
        
        game_data = response.json()
        state = game_data["state"]
        new_score = game_data.get("score", score)
        frame = game_data.get("frame", frame)
        new_available = game_data.get("available_actions", available_actions)
        
        if new_available and hasattr(new_available[0], 'value'):
            available_actions = [a.value for a in new_available]
        else:
            available_actions = new_available
        
        # Store grid state AFTER action
        after_grid = np.array(frame) if isinstance(frame, list) else frame
        
        # =====================================================================
        # REAL-TIME LEARNING: Record causal link
        # =====================================================================
        reward = new_score - score
        
        # Create causal link record
        causal_link = {
            'action': action_name,
            'action_code': action,
            'position': agent.current_position,
            'timestamp': datetime.now().isoformat(),
            'reward': reward,
            'score_before': score,
            'score_after': new_score,
            'changed_cells': int(np.sum(before_grid != after_grid)) if hasattr(before_grid, '__array__') else 0,
        }
        
        # Add to causal tracker (global learning)
        causal_tracker.causal_links.append(causal_link)
        causal_links_this_game.append(causal_link)
        
        # Record in game memory (game-specific learning)
        game_memory.record_action(
            game_exp,
            {'type': action_name, 'code': action, 'pos': agent.current_position},
            before_grid,
            after_grid,
            reward
        )
        
        # =====================================================================
        # REAL-TIME LEARNING: Detect discoveries
        # =====================================================================
        if reward > 0:
            discovery_msg = f"Action {action_name} gave +{reward:.1f} reward"
            print(f"  🎯 DISCOVERY: {discovery_msg}")
            discoveries_this_game += 1
            
            # Try to create a rule from this discovery
            if vlm_available and discoveries_this_game <= 5:  # Limit LLM calls
                new_rule = llm_create_rule(discovery_msg, causal_links_this_game)
                if new_rule:
                    game_memory.add_rule(game_exp, new_rule, is_global=False)
                    game_memory.add_llm_insight(game_exp, discovery_msg)
        
        if causal_link['changed_cells'] > 5:
            print(f"  🔄 Grid changed: {causal_link['changed_cells']} cells")
        
        # Track level wins
        if new_score > score:
            print(f"  ⬆ Level completed! Score: {score} → {new_score}")
            level_wins += 1
            
            # Big discovery - try to induce a global rule
            if vlm_available:
                insight = f"Completed level with action {action_name} at pos {agent.current_position}"
                new_rule = llm_create_rule(insight, causal_links_this_game)
                if new_rule:
                    game_memory.add_rule(game_exp, new_rule, is_global=True)
        
        # =====================================================================
        # REAL-TIME LEARNING: Periodic rule induction
        # =====================================================================
        if actions_taken > 0 and actions_taken % 10 == 0:
            # Try to induce rules from accumulated causal links
            if len(causal_links_this_game) >= 5:
                try:
                    induced = rule_inducer.analyze_causal_links(causal_tracker.causal_links[-50:])
                    if induced:
                        print(f"  📋 Induced {len(induced)} rules from gameplay")
                        for rule in induced[:2]:  # Show first 2
                            game_memory.add_rule(game_exp, str(rule), is_global=True)
                except Exception as e:
                    pass  # Rule induction is optional
        
        score = new_score
        last_action = action
        actions_taken += 1
    
    # =========================================================================
    # END OF GAME: Finalize learning
    # =========================================================================
    
    # Close scorecard
    try:
        session.post(f"{ROOT_URL}/api/scorecard/close", json={"card_id": card_id})
    except:
        pass
    
    # Update game experience
    if score > game_exp.best_score:
        game_exp.best_score = score
    if state == "WIN":
        game_exp.solved = True
        game_exp.solution_path = causal_links_this_game
    
    # Final rule induction from entire game
    if len(causal_links_this_game) >= 10:
        try:
            final_rules = rule_inducer.analyze_causal_links(causal_links_this_game)
            if final_rules:
                for rule in final_rules[:3]:
                    game_memory.add_rule(game_exp, str(rule), is_global=False)
        except:
            pass
    
    # Create spatial memory snapshot
    grid_size = 64
    if agent.last_grid:
        grid_size = max(len(agent.last_grid), len(agent.last_grid[0]) if agent.last_grid else 64)
    
    spatial_snapshot = {
        'exploration_mass': spatial_memory.exploration_mass[:grid_size, :grid_size].clone().cpu().numpy(),
        'curiosity_field': spatial_memory.curiosity_field[:grid_size, :grid_size].clone().cpu().numpy(),
        'grid_size': grid_size,
        'final_pos': agent.current_position,
    }
    
    # Results
    result = {
        'game_id': game_id,
        'card_id': card_id,
        'final_state': state,
        'final_score': score,
        'actions_taken': actions_taken,
        'level_wins': level_wins,
        'agent_stats': agent.get_stats(),
        'spatial_snapshot': spatial_snapshot,
        'learning_stats': {
            'causal_links_recorded': len(causal_links_this_game),
            'discoveries': discoveries_this_game,
            'rules_created': len(game_exp.local_rules),
            'play_count': game_exp.play_count,
            'is_replay': game_exp.play_count > 1,
        }
    }
    
    print(f"\n{'-'*40}")
    print(f"  Result: {state}")
    print(f"  Score: {score}")
    print(f"  Actions: {actions_taken}")
    print(f"  Levels won: {level_wins}")
    print(f"  Effectiveness: {result['agent_stats'].get('effectiveness_rate', 0):.1%}")
    print(f"\n  📊 LEARNING STATS:")
    print(f"     Causal links recorded: {len(causal_links_this_game)}")
    print(f"     Discoveries: {discoveries_this_game}")
    print(f"     Rules created: {len(game_exp.local_rules)}")
    print(f"     LLM insights: {len(game_exp.llm_insights)}")
    print(f"     Total causal links (all games): {len(causal_tracker.causal_links)}")
    print(f"     Scorecard: {ROOT_URL}/scorecards/{card_id}")
    
    return result

print("✓ play_game_with_flux() defined with REAL-TIME LEARNING")
print("\nCapabilities:")
print("  • Records causal links after every action")
print("  • Induces rules every 10 actions")
print("  • Uses LLM to create rules on-the-fly")
print("  • Remembers games for smarter replays")
print("  • Avoids previously failed strategies")

## Cell 10: Run FLUX on ls20

In [ ]:
%%time
# Test on ls20 - Agent Reasoning game
result_ls20 = play_game_with_flux("ls20", max_actions=50, verbose=True)

## Cell 11: Run FLUX on ft09

In [ ]:
%%time
# Test on ft09 - Elementary Logic game
result_ft09 = play_game_with_flux("ft09", max_actions=50, verbose=True)

## Cell 12: Run FLUX on vc33

In [ ]:
%%time
# Test on vc33 - Orchestration game
result_vc33 = play_game_with_flux("vc33", max_actions=50, verbose=True)

## Cell 13: Aggregate Results

In [ ]:
print("\n" + "=" * 60)
print("FLUX UNIFIED AGENT — ARC-AGI-3 TEST RESULTS")
print("=" * 60)

results = [r for r in [result_ls20, result_ft09, result_vc33] if r is not None]

print(f"\nGames tested: {len(results)}")
print(f"\n{'Game':<10} {'Scheme':<18} {'State':<12} {'Score':<8} {'Actions':<8} {'Effect%':<8}")
print("-" * 70)

total_score = 0
total_actions = 0
total_wins = 0

for r in results:
    scheme = r['agent_stats'].get('control_scheme', 'N/A')
    eff_rate = r['agent_stats'].get('effectiveness_rate', 0)
    print(f"{r['game_id']:<10} {scheme:<18} {r['final_state']:<12} {r['final_score']:<8.1f} {r['actions_taken']:<8} {eff_rate:<8.1%}")
    total_score += r['final_score']
    total_actions += r['actions_taken']
    total_wins += r['level_wins']

print("-" * 70)
avg_eff = sum(r['agent_stats'].get('effectiveness_rate', 0) for r in results) / max(1, len(results))
print(f"{'TOTAL':<10} {'':<18} {'':<12} {total_score:<8.1f} {total_actions:<8} {avg_eff:<8.1%}")

print(f"\n\nStep-Aware Cognitive Stats:")
print(f"  Control schemes used: {set(r['agent_stats'].get('control_scheme') for r in results)}")
print(f"  Average effectiveness: {avg_eff:.1%}")
print(f"  Causal links learned: {len(causal_tracker.causal_links)}")
print(f"  Rules induced: {len(rule_inducer.rules)}")

# Check against spec targets
print(f"\n\nSpec Targets:")
print(f"  ls20: {next((r['final_score'] for r in results if r['game_id']=='ls20'), 0):.1f}/0.3")
print(f"  ft09: {next((r['final_score'] for r in results if r['game_id']=='ft09'), 0):.1f}/0.3")
print(f"  vc33: {next((r['final_score'] for r in results if r['game_id']=='vc33'), 0):.1f}/0.3")

## Cell 14: Visualize Spatial Memory

In [ ]:
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# Visualize Spatial Memory for EACH GAME separately
# ─────────────────────────────────────────────

results = [r for r in [result_ls20, result_ft09, result_vc33] if r is not None]

if not results:
    print("No results to visualize!")
else:
    n_games = len(results)
    
    # Create figure with 2 rows per game (mass + ice)
    fig, axes = plt.subplots(n_games, 2, figsize=(14, 5 * n_games))
    
    # Handle single game case (axes won't be 2D)
    if n_games == 1:
        axes = [axes]
    
    for i, result in enumerate(results):
        game_id = result['game_id']
        snap = result.get('spatial_snapshot', {})
        
        mass = snap.get('exploration_mass')
        ice = snap.get('curiosity_field')
        grid_size = snap.get('grid_size', 64)
        final_pos = snap.get('final_pos', (0, 0))
        detected_pos = snap.get('detected_agent_pos')
        clicks = snap.get('clicks_explored', 0)
        
        if mass is None or ice is None:
            print(f"[{game_id}] No spatial data captured")
            continue
        
        # Exploration Mass (Water)
        im1 = axes[i][0].imshow(mass, cmap='hot', vmin=0)
        axes[i][0].set_title(f'{game_id}: Exploration Mass (Water)\nWhere FLUX explored')
        axes[i][0].set_xlabel(f'Final pos: {final_pos} | Detected: {detected_pos}')
        plt.colorbar(im1, ax=axes[i][0])
        
        # Mark final position
        if final_pos:
            axes[i][0].scatter([final_pos[1]], [final_pos[0]], c='cyan', s=100, marker='*', label='Final')
        
        # Curiosity Ice
        im2 = axes[i][1].imshow(ice, cmap='cool', vmin=0)
        axes[i][1].set_title(f'{game_id}: Curiosity Ice\nWhat was interesting')
        axes[i][1].set_xlabel(f'Score: {result["final_score"]} | Actions: {result["actions_taken"]} | Clicks: {clicks}')
        plt.colorbar(im2, ax=axes[i][1])
    
    plt.tight_layout()
    plt.show()
    
    # ─────────────────────────────────────────────
    # ASCII visualizations for each game
    # ─────────────────────────────────────────────
    print("\n" + "=" * 80)
    print("SPATIAL MEMORY ASCII VIEW — Per Game")
    print("=" * 80)
    
    for result in results:
        game_id = result['game_id']
        snap = result.get('spatial_snapshot', {})
        grid_size = snap.get('grid_size', 64)
        final_pos = snap.get('final_pos', (0, 0))
        mass = snap.get('exploration_mass')
        ice = snap.get('curiosity_field')
        
        print(f"\n{'─' * 60}")
        print(f"Game: {game_id}")
        print(f"Score: {result['final_score']} | Actions: {result['actions_taken']} | Final Position: {final_pos}")
        print(f"{'─' * 60}")
        
        if mass is None:
            print("  (No spatial data)")
            continue
        
        # Simple ASCII visualization
        h, w = mass.shape
        display_h = min(h, 32)  # Limit for display
        display_w = min(w, 64)
        
        print()
        for r in range(display_h):
            row_str = ""
            for c in range(display_w):
                m = mass[r, c]
                i = ice[r, c]
                
                if (r, c) == final_pos:
                    row_str += "@ "
                elif m > 10:
                    row_str += "● "  # Heavy exploration
                elif m > 1:
                    row_str += "· "  # Light exploration
                elif i > 10:
                    row_str += "🧊"  # High curiosity
                elif i > 1:
                    row_str += "❄ "  # Low curiosity
                else:
                    row_str += "  "  # Unexplored
            print(row_str)
        
        if h > display_h or w > display_w:
            print(f"  ... (showing {display_h}x{display_w} of {h}x{w})")
        
        print(f"\nLegend: @ final pos | ● explored | · visited | 🧊 high ice | ❄ ice")
        print(f"Stats: {snap.get('clicks_explored', 0)} cells clicked")

## Cell 15: Summary

In [ ]:
print("=" * 60)
print("FLUX UNIFIED AGENT — ARC-AGI-3 LIVE TEST COMPLETE")
print("=" * 60)

print(f"""
Model: {model_path.name}
Agent: FLUXUnifiedAgent (PHASE_UNIFIED_AGENT_SPEC v1.0)
Games tested: {len(results)}
Total levels won: {total_wins}
Total actions: {total_actions}

Step-Aware Cognitive Stack:
  ✓ FrameDiffer (sees what changed)
  ✓ Control-specific strategies
      - MovementStrategy (for ls20-style games)
      - ClickStrategy (for ft09/vc33-style games)
  ✓ SpatialMemory (Ice & Water navigation)
  ✓ CausalTracker ({len(causal_tracker.causal_links)} links learned)
  ✓ RuleInducer ({len(rule_inducer.rules)} rules)
  ✓ GridToWave encoder

Agent Stats:
""")

if results:
    stats = results[-1]['agent_stats']
    for key, val in stats.items():
        if isinstance(val, float):
            print(f"    {key}: {val:.3f}")
        else:
            print(f"    {key}: {val}")

print(f"""
View scorecards at:
""")

for r in results:
    print(f"  {r['game_id']}: {ROOT_URL}/scorecards/{r['card_id']}")

print(f"""

Success Criteria (from spec):
  Target: ls20 >0.3, ft09 >0.3, vc33 >0.3
  Actual: {', '.join(f"{r['game_id']}={r['final_score']}" for r in results)}

Next Steps:
  1. Add Qwen2.5-VL for visual reasoning
  2. Improve rule induction from causal observations
  3. Test on more games (wa30, tr87, sc25, etc.)
  4. Submit to competition (OperationMode.COMPETITION)
""")

## Cell 16: Save Learned Knowledge to Model

**CRITICAL**: This cell saves everything learned during gameplay back to the model:

| Component | What's Saved |
|-----------|-------------|
| `game_memory` | All games played, actions tried, what worked/failed |
| `causal_links` | Every action → effect relationship recorded |
| `rules` | Rules induced from patterns + LLM-created rules |
| `spatial_memory` | Exploration mass and curiosity fields |

**Result**: Next time this model runs, it will:
- Recognize games it's played before
- Avoid strategies that failed
- Follow promising directions from previous runs
- Apply learned rules automatically

In [ ]:
# =============================================================================
# SAVE ALL LEARNED KNOWLEDGE TO MODEL
# =============================================================================

print("=" * 60)
print("💾 SAVING LEARNED KNOWLEDGE TO MODEL")
print("=" * 60)

# ─────────────────────────────────────────────
# 1. Save Game Memory (per-game experience)
# ─────────────────────────────────────────────
print("\n1. Saving Game Memory...")
game_memory.save_to_model()
print(f"   • Games remembered: {len(game_memory.games)}")
print(f"   • Global rules: {len(game_memory.global_rules)}")

# ─────────────────────────────────────────────
# 2. Update Causal Tracker in model
# ─────────────────────────────────────────────
print("\n2. Updating Causal Tracker...")
existing_links = flux_model.get('causal_tracker', {}).get('causal_links', [])
new_links = causal_tracker.causal_links

# Merge (append new links, avoid duplicates by timestamp)
existing_timestamps = {l.get('timestamp') for l in existing_links if isinstance(l, dict)}
unique_new = [l for l in new_links if l.get('timestamp') not in existing_timestamps]

all_links = existing_links + unique_new
flux_model.set('causal_tracker', {
    'causal_links': all_links[-1000:],  # Keep last 1000 links
    'total_recorded': len(all_links),
    'last_updated': datetime.now().isoformat(),
})
print(f"   • Total causal links: {len(all_links)}")
print(f"   • New links added: {len(unique_new)}")

# ─────────────────────────────────────────────
# 3. Update Rule Inducer in model
# ─────────────────────────────────────────────
print("\n3. Updating Rule Inducer...")
existing_rules = flux_model.get('rule_inducer', {}).get('rules', [])

# Get rules from rule_inducer object
new_rules = []
if hasattr(rule_inducer, 'rules'):
    for rule in rule_inducer.rules:
        if hasattr(rule, '__dict__'):
            new_rules.append(str(rule))
        else:
            new_rules.append(rule)

# Also add game memory global rules
for rule in game_memory.global_rules:
    if rule not in new_rules and rule not in existing_rules:
        new_rules.append(rule)

all_rules = list(set(existing_rules + new_rules))  # Deduplicate
flux_model.set('rule_inducer', {
    'rules': all_rules,
    'total_rules': len(all_rules),
    'last_updated': datetime.now().isoformat(),
})
print(f"   • Total rules: {len(all_rules)}")
print(f"   • New rules added: {len(new_rules)}")

# ─────────────────────────────────────────────
# 4. Update Spatial Memory snapshot
# ─────────────────────────────────────────────
print("\n4. Updating Spatial Memory...")
spatial_data = {
    'exploration_mass': spatial_memory.exploration_mass.cpu().numpy().tolist(),
    'curiosity_field': spatial_memory.curiosity_field.cpu().numpy().tolist(),
    'total_explored': int(torch.sum(spatial_memory.exploration_mass > 0).item()),
    'last_updated': datetime.now().isoformat(),
}
flux_model.set('spatial_memory', spatial_data)
print(f"   • Cells explored: {spatial_data['total_explored']}")

# ─────────────────────────────────────────────
# 5. Save model to disk and HF Hub
# ─────────────────────────────────────────────
print("\n5. Saving enhanced model...")

# Add session metadata
flux_model.set('last_session', {
    'timestamp': datetime.now().isoformat(),
    'games_played': [g.game_id for g in game_memory.games.values()],
    'total_actions': sum(len(g.actions_tried) for g in game_memory.games.values()),
    'total_discoveries': len(game_memory.discovery_log),
})

# Save locally
local_save_path = Path('/kaggle/working/Flux-multi-model-enhanced.flx')
flux_model.save(str(local_save_path))
print(f"   ✓ Saved locally: {local_save_path}")
print(f"     Size: {local_save_path.stat().st_size / 1024 / 1024:.2f} MB")

# Upload to HuggingFace Hub
print("\n6. Uploading to HuggingFace Hub...")
try:
    from huggingface_hub import HfApi
    api = HfApi()
    
    api.upload_file(
        path_or_fileobj=str(local_save_path),
        path_in_repo="Flux-multi-model.flx",
        repo_id="UnseenGAP/FLUX",
        token=HF_TOKEN,
        commit_message=f"ARC-AGI-3 learning session: {len(game_memory.games)} games, {len(all_links)} causal links, {len(all_rules)} rules"
    )
    print(f"   ✓ Uploaded to UnseenGAP/FLUX")
except Exception as e:
    print(f"   ⚠ HF upload failed: {e}")
    print(f"   (Model saved locally, upload manually later)")

# ─────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("✅ LEARNING SAVED SUCCESSFULLY")
print("=" * 60)
print(f"""
What's now in the model:
  📁 Game Memory: {len(game_memory.games)} games remembered
  🔗 Causal Links: {len(all_links)} action→effect relationships  
  📋 Rules: {len(all_rules)} learned rules
  🗺️ Spatial Memory: {spatial_data['total_explored']} cells explored

Next run will:
  • Recognize previously played games
  • Apply successful strategies automatically
  • Avoid actions that failed before
  • Use learned rules for faster solving
""")

## Cell 17: Replay Demo — Test Learning Persistence

**Demonstration**: Play the same game again to show that the agent:
1. Recognizes it played this game before
2. Shows what it learned last time
3. Tries different strategies based on previous experience
4. Avoids actions that previously failed

In [ ]:
# =============================================================================
# REPLAY DEMO: Show learning in action
# =============================================================================
# Run this cell AFTER using the save cell above to demonstrate learning persistence

print("=" * 60)
print("🔄 REPLAY DEMO — Testing Learning Persistence")
print("=" * 60)

# Pick a game we already played
replay_game = "ls20"  # or "ft09" or "vc33"

print(f"\nReplaying: {replay_game}")
print("Watch for:")
print("  • '🔁 RECOGNIZED GAME!' message (proves we remember)")
print("  • Previous experience summary (what worked/failed)")
print("  • Different strategy based on experience")
print()

# Replay with lower action limit to see quicker result
result_replay = play_game_with_flux(replay_game, max_actions=30, verbose=True)

if result_replay:
    game_exp = game_memory.games.get(replay_game)
    if game_exp:
        print("\n" + "=" * 60)
        print("📊 LEARNING COMPARISON")
        print("=" * 60)
        print(f"  Play count: {game_exp.play_count}")
        print(f"  Best score ever: {game_exp.best_score}")
        print(f"  Actions tried total: {len(game_exp.actions_tried)}")
        print(f"  Successful actions: {len(game_exp.successful_actions)}")
        print(f"  Failed actions: {len(game_exp.failed_actions)}")
        print(f"  Local rules learned: {len(game_exp.local_rules)}")
        
        if game_exp.promising_directions:
            print(f"\n  ✓ Promising directions: {game_exp.promising_directions[:5]}")
        if game_exp.avoid_directions:
            print(f"  ✗ Avoiding: {game_exp.avoid_directions[:5]}")
        
        print("\nThe agent now knows this game and will do better each time!")